# Codebase Workflow Walkthrough

This notebook is a guided tour of the live `src/` code.

It is designed to answer three questions clearly:

1. What is the workflow of the code?
2. What does each layer do?
3. What is already implemented and working?

We start from the structured model layer and move upward through gauge metadata, gauge compilation, and the convention-fixed covariant and pure-gauge features.

It covers the main workflow of the codebase, but not every regression case that lives in `examples/examples.py`.

## Recommended way to read this notebook

Run the notebook from top to bottom.

The order is intentional:

- understand the public declaration layer
- build a first `Model`
- compile covariant kinetic terms
- inspect repeated-slot scalar internals
- compile pure-gauge Yang-Mills structures

That mirrors the actual growth of the codebase.

Quick orientation while reading:

- when the notebook uses `Field`, `Model(...)`, `model.lagrangian()`, or `feynman_rule(...)`, it is showing the modern public API from `src/model/`
- when the notebook calls `vertex_factor(...)` or constructs manual legs later on, it is intentionally dropping into backend internals in `src/symbolic/vertex_engine.py`
- when the notebook calls `compile_complex_scalar_gauge_terms(...)`, it enters internal lowering in `src/compiler/gauge.py`


In [1]:
import sys
from pathlib import Path
from fractions import Fraction

root = Path.cwd()
src_path = root / "src"
if not src_path.exists():
    src_path = root.parent / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from symbolic.vertex_engine import (
    S,
    Expression,
    I,
    Dot,
    vertex_factor,
    simplify_deltas,
)

from symbolic.spenso_structures import (
    SPINOR_KIND,
    LORENTZ_KIND,
    COLOR_FUND_KIND,
    COLOR_ADJ_KIND,
    gauge_generator,
    structure_constant,
    simplify_gamma_chain,
    weak_gauge_generator, weak_structure_constant,
)

from model import (
    Field,
    GaugeGroup,
    GaugeRepresentation,
    Model,
    CompiledLagrangian,
    CovD,
    PartialD,
    Gamma,
    FieldStrength,
    GaugeFixing,
    GhostLagrangian,
    SPINOR_INDEX,
    LORENTZ_INDEX,
    COLOR_FUND_INDEX,
    COLOR_ADJ_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
)



from compiler.gauge import (
    compile_complex_scalar_gauge_terms,
)

print("Python:", sys.executable)
print("src path:", src_path.resolve())

x = S("x")
d = S("d")


def model_vertex(interaction, external_legs, species_map=None, strip_externals=True, include_delta=False, simplify_gamma=False):
    expr = vertex_factor(
        interaction=interaction,
        external_legs=external_legs,
        x=x,
        d=d,
        strip_externals=strip_externals,
        include_delta=include_delta,
    )
    expr = simplify_deltas(expr, species_map=species_map)
    q_ = S("q_")
    x_ = S("x_")
    expr = expr.replace(Expression.EXP(-I * Dot(q_, x_)), 1)
    if simplify_gamma:
        expr = simplify_gamma_chain(expr)
    return expr


def show(title, expr):
    print()
    print(f"=== {title} ===")
    print(expr)


def labels(terms):
    return [term.label for term in terms]


Python: /Users/rems/Documents/MScThesis/.venv/bin/python
src path: /Users/rems/Documents/MScThesis/src


## 0. Workflow in one paragraph

The public workflow is `Model(...) -> model.lagrangian() -> feynman_rule(...)`.

Under that public API, the current layers are:

- `src/model/`: `Field`, `Model`, declarative source terms, and public extraction helpers
- `src/compiler/gauge.py`: the lowering from source declarations to backend compiled interactions
- `src/symbolic/vertex_engine.py`: `vertex_factor(...)`, simplification helpers, and derivative bookkeeping

Internally, compilation still produces backend interaction objects (`InteractionTerm`), and the symbolic engine evaluates those compiled terms.

So the notebook alternates between two questions:

- what source declaration should a user write?
- what backend compiled term did that declaration become?


In [2]:
p1, p2, p3, p4 = S("p1", "p2", "p3", "p4")
b1, b2, b3, b4 = S("b1", "b2", "b3", "b4")

phi0, chi0 = S("phi0", "chi0")
phiQED0, phiQEDdag0 = S("phiQED0", "phiQEDdag0")
phiQCD0, phiQCDdag0 = S("phiQCD0", "phiQCDdag0")
psi0, psibar0 = S("psi0", "psibar0")
psiQED0, psibarQED0 = S("psiQED0", "psibarQED0")
A0, G0 = S("A0", "G0")

mu, nu, rho, sigma = S("mu", "nu", "rho", "sigma")
mu3, mu4 = S("mu3", "mu4")

lam4, yF, eQED, qPsi, qPhi, gS = S(
    "lam4", "yF", "eQED", "qPsi", "qPhi", "gS"
)

alpha_s = S("alpha_s")
i1, i2 = S("i1", "i2")
s1, s2 = S("s1", "s2")
a3, a4, a5, a6 = S("a3", "a4", "a5", "a6")
c1, c2 = S("c1", "c2")

print("Common symbols initialised.")


Common symbols initialised.


## 1. First public objects: `Field` and `Model`

The modern API starts from declarative source expressions, not from handwritten `FieldOccurrence`, `ExternalLeg`, or `InteractionTerm` objects.

The core user-facing objects here are:

- `Field`: one declared particle field, with spin/statistics/index metadata
- `Model`: the top-level container that stores a source declaration and compiles it on demand

A local monomial such as `lam4 * Phi * Phi * Phi * Phi` or `yF * Psi.bar * Psi * Phi` is already valid source input to `Model(...)`.

Later in this notebook we will inspect backend compiled terms explicitly. That is internal debugging/introspection, not the recommended declaration path.


In [ ]:
PhiField = Field("Phi", spin=0, self_conjugate=True, symbol=phi0)
ChiField = Field("Chi", spin=0, self_conjugate=True, symbol=chi0)
PsiField = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX,),
)
GaugeField = Field("A", spin=1, self_conjugate=True, symbol=A0, indices=(LORENTZ_INDEX,))

print("Phi role:", PhiField.role_for(False))
print("Psi species / conjugate species:", PsiField.species_for(False), "/", PsiField.species_for(True))
print("Gauge index kinds:", [index.kind for index in GaugeField.indices])

LOCAL_PHI4_MODEL = Model(
    lam4 * PhiField * PhiField * PhiField * PhiField,
    name="local phi^4",
)
LOCAL_YUKAWA_MODEL = Model(
    yF * PsiField.bar * PsiField * PhiField,
    name="local Yukawa",
)

print()
print("Local phi^4 declaration:", LOCAL_PHI4_MODEL.lagrangian_decl)
print("Local Yukawa declaration:", LOCAL_YUKAWA_MODEL.lagrangian_decl)

show(
    "phi^4 through the public API",
    LOCAL_PHI4_MODEL.feynman_rule(
        PhiField,
        PhiField,
        PhiField,
        PhiField,
        momenta=(p1, p2, p3, p4),
    ),
)
show(
    "Yukawa through the public API",
    LOCAL_YUKAWA_MODEL.feynman_rule(
        PsiField.bar,
        PsiField,
        PhiField,
        momenta=(p1, p2, p3),
    ),
)


## 2. First `Model` object

A `Model` is the top-level container for a theory.

At minimum it stores:

- declared source terms
- inferred or explicit fields
- gauge groups
- parameters

Internally, `model.lagrangian()` compiles those source terms into backend interaction objects, but the public workflow stays at the declaration level.


In [ ]:
ToyScalarModel = Model(
    lam4 * PhiField * PhiField * PhiField * PhiField,
    name="Toy scalar model",
    fields=(PhiField, ChiField),
)

compiled_toy = ToyScalarModel.lagrangian()

print("Model name:", ToyScalarModel.name)
print("Declared source:", ToyScalarModel.lagrangian_decl)
print("Stored fields:", [field.name for field in ToyScalarModel.fields])
print("find_field('Phi') ->", ToyScalarModel.find_field("Phi"))
print("Compiled term count ->", len(compiled_toy.terms))


## 3. Gauge metadata: `GaugeRepresentation` and `GaugeGroup`

To move beyond handwritten interaction terms, the code introduces gauge metadata.

- `GaugeRepresentation` tells the compiler how to build the generator tensor for one matter representation
- `GaugeGroup` defines the symmetry itself: coupling, gauge boson, structure constants, charge label, and supported matter representations

This is the basis for the gauge compiler.

In [5]:
PhiQEDField = Field(
    "PhiQED",
    spin=0,
    self_conjugate=False,
    symbol=phiQED0,
    conjugate_symbol=phiQEDdag0,
    quantum_numbers={"Q": qPhi},
)

PsiQEDField = Field(
    "PsiQED",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psiQED0,
    conjugate_symbol=psibarQED0,
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)

PhiQCDField = Field(
    "PhiQCD",
    spin=0,
    self_conjugate=False,
    symbol=phiQCD0,
    conjugate_symbol=phiQCDdag0,
    indices=(COLOR_FUND_INDEX,),
)

QuarkField = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)

GluonField = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=G0,
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)

ghG0, ghGbar0 = S("ghG0", "ghGbar0")
GhostGluonField = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=ghG0,
    conjugate_symbol=ghGbar0,
    indices=(COLOR_ADJ_INDEX,),
)

COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental",
)

QED_GROUP = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson=GaugeField,
    charge="Q",
)

QCD_GROUP = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=GluonField,
    ghost_field=GhostGluonField,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_QED_BASE = Model(
    name="FermionQED-minimal",
    gauge_groups=(QED_GROUP,),
    fields=(PsiQEDField, GaugeField),
)

MODEL_QCD_BASE = Model(
    name="QCD-minimal",
    gauge_groups=(QCD_GROUP,),
    fields=(QuarkField, GluonField),
)

print("QCD representation seen by the quark ->", QCD_GROUP.matter_representation(QuarkField))
print("Gauge boson resolved for QED ->", MODEL_QED_BASE.gauge_boson_field(QED_GROUP))


QCD representation seen by the quark -> GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', dimension=None, is_flavor=False, prefix='c'), generator_builder=<function gauge_generator at 0x10ce95010>, name='fundamental', slot=None, slot_policy='unique')
Gauge boson resolved for QED -> Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={}, ghost_of=None, flavor_index=None, class_members=())


## 4. Repeated same-kind index slots

This section inspects repeated-slot behavior through an internal scalar helper that still exists under the modern `Model(...)` pipeline.

This uses:

- `Field(...)` and `GaugeRepresentation(slot=...)` from the `src/model/` package
- `compile_complex_scalar_gauge_terms(...)` from `src/compiler/gauge.py`

If the representation is ambiguous, the compiler now rejects the model instead of silently collapsing repeated slots.


In [7]:
phiBi0, phiBidag0 = S("phiBi0", "phiBidag0")
c3, c4 = S("c3", "c4")

PhiBiField = Field(
    "PhiBi",
    spin=0,
    self_conjugate=False,
    symbol=phiBi0,
    conjugate_symbol=phiBidag0,
    indices=(COLOR_FUND_INDEX, COLOR_FUND_INDEX),
)

COLOR_FUND_REP_SLOT0 = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental_slot0",
    slot=0,
)

QCD_GROUP_BISLOT = GaugeGroup(
    name="SU3CBi",
    abelian=False,
    coupling=gS,
    gauge_boson=GluonField,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP_SLOT0,),
)

QCD_GROUP_AMBIGUOUS = GaugeGroup(
    name="SU3CAmbiguous",
    abelian=False,
    coupling=gS,
    gauge_boson=GluonField,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_BISLOT_OK = Model(
    name="ScalarQCD-bislot-minimal",
    gauge_groups=(QCD_GROUP_BISLOT,),
    fields=(PhiBiField, GluonField),
)

MODEL_BISLOT_BAD = Model(
    name="ScalarQCD-bislot-ambiguous",
    gauge_groups=(QCD_GROUP_AMBIGUOUS,),
    fields=(PhiBiField, GluonField),
)

try:
    compile_complex_scalar_gauge_terms(
        scalar=PhiBiField,
        gauge_group=QCD_GROUP_AMBIGUOUS,
        gauge_field=GluonField,
    )
except ValueError as exc:
    print("Ambiguous repeated-slot model rejected:")
    print(" ", exc)

compiled_bislot = compile_complex_scalar_gauge_terms(
    scalar=PhiBiField,
    gauge_group=QCD_GROUP_BISLOT,
    gauge_field=GluonField,
)
current_terms = tuple(term for term in compiled_bislot if "current" in term.label)
print("Repeated-slot current labels:", labels(current_terms))

LEGS_bislot = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)

V_bislot = sum(
    (
        model_vertex(
            term,
            LEGS_bislot,
            species_map={b1: phiBidag0, b2: phiBi0, b3: G0},
        )
        for term in current_terms
    ),
    Expression.num(0),
)

show("Repeated-slot scalar gauge current (slot 1 active, slot 2 spectator)", V_bislot)


Ambiguous repeated-slot model rejected:
  Field 'PhiBi' carries repeated index type 'ColorFund'; declare GaugeRepresentation(slot=...) for strict selection, or set GaugeRepresentation(slot_policy='sum') to sum over all matching slots.
Repeated-slot current labels: ['SU3CBi: scalar current (+)_slot1', 'SU3CBi: scalar current (-)_slot1']

=== Repeated-slot scalar gauge current (slot 1 active, slot 2 spectator) ===
-gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)


## 5. Declarative kinetic sectors via `lagrangian_decl`

The recommended front-end is now one unified declaration entry point:

- `Model(...)`

For the standard gauge-generated pieces, use canonical declarative objects:

- `I * Psi.bar * Gamma(mu) * CovD(Psi, mu)`
- `CovD(Phi.bar, mu) * CovD(Phi, mu)`
- `-(1/4) * FieldStrength(G, mu, nu) * FieldStrength(G, mu, nu)`

For ordinary local derivative operators, use `PartialD(...)` inside the same
source monomial language.

The important implementation point is now simple: `lagrangian_decl` stores source
terms, and `Model.lagrangian()` inspects them one by one. Pure local monomials compile
directly into backend interaction terms, while terms containing canonical `CovD`,
`FieldStrength`, gauge-fixing, or ghost structure are expanded through the compiler.


In [8]:
mu_decl = S("mu_decl")

MODEL_QED_FERMION_COV = Model(
    I * PsiQEDField.bar * Gamma(mu_decl) * CovD(PsiQEDField, mu_decl),
    name="FermionQED-covariant",
    gauge_groups=(QED_GROUP,),
)

print("Declared fermion QED Lagrangian:", MODEL_QED_FERMION_COV.lagrangian_decl)
compiled_qed_cov = MODEL_QED_FERMION_COV.lagrangian().terms
print("Covariant fermion-QED terms:", labels(compiled_qed_cov))

LEGS_qed_decl = (
    PsiQEDField.leg(p1, conjugated=True, labels={SPINOR_KIND: i1}),
    PsiQEDField.leg(p2, labels={SPINOR_KIND: i2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}),
)

V_qed_cov = model_vertex(compiled_qed_cov[0], LEGS_qed_decl, simplify_gamma=True)
print("=====Dirac kinetic term (QED) via Model(...)=====")
show("Vertex qbar q A from I * psibar * Gamma(mu) * CovD(psi, mu)", V_qed_cov)

MODEL_SCALAR_QED_COV = Model(
    CovD(PhiQEDField.bar, mu_decl) * CovD(PhiQEDField, mu_decl),
    name="ScalarQED-covariant",
    gauge_groups=(QED_GROUP,),
)

print("Declared scalar QED Lagrangian:", MODEL_SCALAR_QED_COV.lagrangian_decl)
compiled_sqed = MODEL_SCALAR_QED_COV.lagrangian().terms
print("Covariant scalar-QED terms:", labels(compiled_sqed))

LEGS_sqed_current = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
)
LEGS_sqed_contact = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
    GaugeField.leg(p4, labels={LORENTZ_KIND: mu4}, species=b4),
)

sqed_species_3 = {b1: phiQEDdag0, b2: phiQED0, b3: A0}
sqed_species_4 = {b1: phiQEDdag0, b2: phiQED0, b3: A0, b4: A0}

V_sqed_current = sum(
    (model_vertex(t, LEGS_sqed_current, species_map=sqed_species_3) for t in compiled_sqed if "current" in t.label),
    Expression.num(0),
).expand()
V_sqed_contact = sum(
    (model_vertex(t, LEGS_sqed_contact, species_map=sqed_species_4) for t in compiled_sqed if "contact" in t.label),
    Expression.num(0),
).expand()

print("=====Complex-scalar kinetic term (QED) via Model(...) — current=====")
show("Vertex phi^dagger phi A from CovD(phi.bar, mu) * CovD(phi, mu)", V_sqed_current)
print("=====Complex-scalar kinetic term (QED) via Model(...) — contact=====")
show("Vertex phi^dagger phi A A from CovD(phi.bar, mu) * CovD(phi, mu)", V_sqed_contact)


Declared fermion QED Lagrangian: 1𝑖 * PsiQED.bar * Gamma(mu_decl) * CovD(PsiQED, mu_decl)
Covariant fermion-QED terms: ['i PsiQEDbar gamma^mu D_mu PsiQED', 'i PsiQEDbar gamma^mu D_mu PsiQED partial']
=====Dirac kinetic term (QED) via Model(...)=====

=== Vertex qbar q A from I * psibar * Gamma(mu) * CovD(psi, mu) ===
1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
Declared scalar QED Lagrangian: CovD(PhiQED.bar, mu_decl) * CovD(PhiQED, mu_decl)
Covariant scalar-QED terms: ['(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (+)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (-)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar contact', '(D_mu PhiQED)^dagger (D^mu PhiQED) derivative']
=====Complex-scalar kinetic term (QED) via Model(...) — current=====

=== Vertex phi^dagger phi A from CovD(phi.bar, mu) * CovD(phi, mu) ===
-1𝑖*eQED*qPhi*pcomp(p1,mu)+1𝑖*eQED*qPhi*pcomp(p2,mu)
=====Complex-scalar kinetic term (QED) via Model(...) — contact=====

=== Vertex phi

## 5.1 Covariant derivative compiler (matter kinetic terms)

In this codebase, `CovD(...)` means the gauge-theory covariant derivative.

### Conventions (frozen for the physical compiler)

- **Fourier-space derivative rule**: `partial_mu -> -i p_mu`
- **Overall vertex factor**: `vertex_factor(...)` contributes the universal `+i`
- **Matter covariant derivative**:

\[
D_\mu = \partial_\mu + i g A_\mu
\]

### What `Model.lagrangian()` now lowers

The declarative front-end lowers the covered canonical forms:

- `I * Psi.bar * Gamma(mu) * CovD(Psi, mu)`
- `CovD(Phi.bar, mu) * CovD(Phi, mu)`

into ordinary compiled interaction terms.

The backend representation is `InteractionTerm`, but the public workflow remains `Model(...)` plus `feynman_rule(...)`.

### Repeated same-kind index slots and `slot_policy`

If a field carries the same index type more than once, gauge generators can be
ambiguous.

Rule:

- ambiguity -> error by default (`slot_policy="unique"`)
- explicit opt-in for tensor-product semantics (`slot_policy="sum"`)


In [9]:
# --- Fermion QCD: I * qbar * Gamma(mu) * CovD(q, mu) ---
mu_qcd_decl = S("mu_qcd_decl")
MODEL_QCD_FERMION_COV = Model(
    I * QuarkField.bar * Gamma(mu_qcd_decl) * CovD(QuarkField, mu_qcd_decl),
    name="QCD-covariant",
    gauge_groups=(QCD_GROUP,),
)

compiled_qcd_cov = MODEL_QCD_FERMION_COV.lagrangian().terms
print("Declared QCD fermion Lagrangian:", MODEL_QCD_FERMION_COV.lagrangian_decl)
print("Generated terms:", labels(compiled_qcd_cov))

LEGS_qcd_decl = (
    QuarkField.leg(p1, conjugated=True, species=b1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    QuarkField.leg(p2, species=b2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)

V_qcd_cov = model_vertex(compiled_qcd_cov[0], LEGS_qcd_decl, species_map={b1: psibar0, b2: psi0, b3: G0}, simplify_gamma=True)
show("QCD quark-gluon vertex from I * qbar * Gamma(mu) * CovD(q, mu)", V_qcd_cov)


Declared QCD fermion Lagrangian: 1𝑖 * q.bar * Gamma(mu_qcd_decl) * CovD(q, mu_qcd_decl)
Generated terms: ['i qbar gamma^mu D_mu q [ColorFund]', 'i qbar gamma^mu D_mu q partial']

=== QCD quark-gluon vertex from I * qbar * Gamma(mu) * CovD(q, mu) ===
1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))


In [10]:
# --- Scalar QCD: CovD(PhiQCD.bar, mu) * CovD(PhiQCD, mu) ---
mu_sqcd_decl = S("mu_sqcd_decl")
MODEL_SCALAR_QCD_COV = Model(
    CovD(PhiQCDField.bar, mu_sqcd_decl) * CovD(PhiQCDField, mu_sqcd_decl),
    name="ScalarQCD-covariant",
    gauge_groups=(QCD_GROUP,),
)

compiled_sqcd = MODEL_SCALAR_QCD_COV.lagrangian().terms
print("Declared scalar QCD Lagrangian:", MODEL_SCALAR_QCD_COV.lagrangian_decl)
print("Generated terms:", labels(compiled_sqcd))

LEGS_sqcd_current = (
    PhiQCDField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiQCDField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)
LEGS_sqcd_contact = (
    PhiQCDField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiQCDField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p4, species=b4, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}),
)

sqcd_species_3 = {b1: phiQCDdag0, b2: phiQCD0, b3: G0}
sqcd_species_4 = {b1: phiQCDdag0, b2: phiQCD0, b3: G0, b4: G0}

V_sqcd_current = sum(
    (model_vertex(t, LEGS_sqcd_current, species_map=sqcd_species_3) for t in compiled_sqcd if "current" in t.label),
    Expression.num(0),
).expand()
V_sqcd_contact = sum(
    (model_vertex(t, LEGS_sqcd_contact, species_map=sqcd_species_4) for t in compiled_sqcd if "contact" in t.label),
    Expression.num(0),
).expand()

show("Scalar QCD current from CovD(Phi.bar, mu) * CovD(Phi, mu)", V_sqcd_current)
show("Scalar QCD contact from CovD(Phi.bar, mu) * CovD(Phi, mu)", V_sqcd_contact)


Declared scalar QCD Lagrangian: CovD(PhiQCD.bar, mu_sqcd_decl) * CovD(PhiQCD, mu_sqcd_decl)
Generated terms: ['(D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar current (+)', '(D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar current (-)', '(D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar contact [slots 1,1]', '(D_mu PhiQCD)^dagger (D^mu PhiQCD) derivative']

=== Scalar QCD current from CovD(Phi.bar, mu) * CovD(Phi, mu) ===
-1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)

=== Scalar QCD contact from CovD(Phi.bar, mu) * CovD(Phi, mu) ===
1𝑖*gS^2*g(mink(4, mu3),mink(4, mu4))*t(coad(8, a3),cof(3, c1),cof(3, c_mid_PhiQCD_SU3C))*t(coad(8, a4),cof(3, c_mid_PhiQCD_SU3C),cof(3, c2))+1𝑖*gS^2*g(mink(4, mu3),mink(4, mu4))*t(coad(8, a3),cof(3, c_mid_PhiQCD_SU3C),cof(3, c2))*t(coad(8, a4),cof(3, c1),cof(3, c_mid_PhiQCD_SU3C))


In [11]:
# --- Mixed QCD + QED fermion kinetic term expands into separate gauge pieces ---
qMix = S("qMix")
PsiMixField = Field(
    "PsiMix",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psiMix0"),
    conjugate_symbol=S("psibarMix0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qMix},
)

mu_mix_decl = S("mu_mix_decl")
MODEL_MIXED_FERMION_COV = Model(
    I * PsiMixField.bar * Gamma(mu_mix_decl) * CovD(PsiMixField, mu_mix_decl),
    name="MixedFermionQCDQED-covariant",
    gauge_groups=(QCD_GROUP, QED_GROUP),
)

compiled_mixed = MODEL_MIXED_FERMION_COV.lagrangian().terms
print("Declared mixed fermion Lagrangian:", MODEL_MIXED_FERMION_COV.lagrangian_decl)
print("Generated terms:")
for lab in labels(compiled_mixed):
    print(" -", lab)

LEGS_mixed_gluon = (
    PsiMixField.leg(p1, conjugated=True, species=b1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    PsiMixField.leg(p2, species=b2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)
LEGS_mixed_photon = (
    PsiMixField.leg(p1, conjugated=True, species=b1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    PsiMixField.leg(p2, species=b2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GaugeField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3}),
)

mixed_species_gluon = {b1: S("psibarMix0"), b2: S("psiMix0"), b3: G0}
mixed_species_photon = {b1: S("psibarMix0"), b2: S("psiMix0"), b3: A0}

# For the mixed fermion case the current labels do not carry the gauge-group name,
# so we use the generated term order: non-abelian piece first, abelian piece second.
V_mixed_gluon = model_vertex(compiled_mixed[0], LEGS_mixed_gluon, species_map=mixed_species_gluon, simplify_gamma=True)
V_mixed_photon = model_vertex(compiled_mixed[1], LEGS_mixed_photon, species_map=mixed_species_photon, simplify_gamma=True)

show("Mixed fermion gluon vertex", V_mixed_gluon)
show("Mixed fermion photon vertex", V_mixed_photon)


Declared mixed fermion Lagrangian: 1𝑖 * PsiMix.bar * Gamma(mu_mix_decl) * CovD(PsiMix, mu_mix_decl)
Generated terms:
 - i PsiMixbar gamma^mu D_mu PsiMix [ColorFund]
 - i PsiMixbar gamma^mu D_mu PsiMix
 - i PsiMixbar gamma^mu D_mu PsiMix partial

=== Mixed fermion gluon vertex ===
1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

=== Mixed fermion photon vertex ===
1𝑖*eQED*qMix*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


### Mixed-group complex scalar kinetic term

The scalar analogue of the mixed QCD+QED fermion example uses the same
`lagrangian_decl` entry point.

For one complex scalar carrying both an SU(3) representation and a U(1) charge,
`CovD(Phi.bar, mu) * CovD(Phi, mu)` generates:

- the SU(3) current
- the U(1) current
- the mixed SU(3) x U(1) contact term


In [12]:
# --- Mixed QCD + QED complex scalar kinetic term: currents + mixed contact ---
phiMix0, phiMixdag0 = S("phiMix0", "phiMixdag0")
qPhiMix = S("qPhiMix")
PhiMixField = Field(
    "PhiMix",
    spin=0,
    self_conjugate=False,
    symbol=phiMix0,
    conjugate_symbol=phiMixdag0,
    indices=(COLOR_FUND_INDEX,),
    quantum_numbers={"Q": qPhiMix},
)
MODEL_MIXED_SCALAR_COV = Model(
    CovD(PhiMixField.bar, mu_decl) * CovD(PhiMixField, mu_decl),
    name="MixedScalarQCDQED-covariant",
    gauge_groups=(QCD_GROUP, QED_GROUP),
)
compiled_mixed_scalar = MODEL_MIXED_SCALAR_COV.lagrangian().terms
print("Declared mixed scalar Lagrangian:", MODEL_MIXED_SCALAR_COV.lagrangian_decl)
print("Generated terms:")
for lab in labels(compiled_mixed_scalar):
    print(" -", lab)

mixed_scalar_qcd_terms = [t for t in compiled_mixed_scalar if "SU3C: scalar current" in t.label]
mixed_scalar_qed_terms = [t for t in compiled_mixed_scalar if "U1QED: scalar current" in t.label]
mixed_scalar_contact_terms = [t for t in compiled_mixed_scalar if "mixed contact" in t.label]

LEGS_mixed_scalar_gluon = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)
LEGS_mixed_scalar_photon = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
)
LEGS_mixed_scalar_contact = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
    GaugeField.leg(p4, labels={LORENTZ_KIND: mu4}, species=b4),
)

mixed_scalar_species_gluon = {b1: phiMixdag0, b2: phiMix0, b3: G0}
mixed_scalar_species_photon = {b1: phiMixdag0, b2: phiMix0, b3: A0}
mixed_scalar_species_contact = {b1: phiMixdag0, b2: phiMix0, b3: G0, b4: A0}

V_mixed_scalar_gluon = sum(
    (model_vertex(t, LEGS_mixed_scalar_gluon, species_map=mixed_scalar_species_gluon) for t in mixed_scalar_qcd_terms),
    Expression.num(0),
).expand()
V_mixed_scalar_photon = sum(
    (model_vertex(t, LEGS_mixed_scalar_photon, species_map=mixed_scalar_species_photon) for t in mixed_scalar_qed_terms),
    Expression.num(0),
).expand()
V_mixed_scalar_contact = sum(
    (model_vertex(t, LEGS_mixed_scalar_contact, species_map=mixed_scalar_species_contact) for t in mixed_scalar_contact_terms),
    Expression.num(0),
).expand()

show("Mixed scalar gluon current", V_mixed_scalar_gluon)
show("Mixed scalar photon current", V_mixed_scalar_photon)
show("Mixed scalar SU(3) x U(1) contact", V_mixed_scalar_contact)


Declared mixed scalar Lagrangian: CovD(PhiMix.bar, mu_decl) * CovD(PhiMix, mu_decl)
Generated terms:
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar current (+)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar current (-)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar contact [slots 1,1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar current (+)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar current (-)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar contact
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C x U1QED: scalar mixed contact [slot 1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED x SU3C: scalar mixed contact [slot 1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) derivative

=== Mixed scalar gluon current ===
-1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)

=== Mixed scalar photon current ===
-1𝑖*eQED*qPhiMix*g(cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*eQED*qPhiMix*g(cof(3, c1),cof(3, c2))*pcomp(p2,mu)

=== M

In [13]:
# --- Bislotted scalar with explicit opt-in: slot_policy='sum' ---
COLOR_FUND_REP_SUM = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental_sum",
    slot_policy="sum",
)
QCD_GROUP_BISLOT_SUM = GaugeGroup(
    name="SU3CBiSum",
    abelian=False,
    coupling=gS,
    gauge_boson=GluonField,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP_SUM,),
)
MODEL_BISLOT_SUM_COV = Model(
    CovD(PhiBiField.bar, S("mu_bis_decl")) * CovD(PhiBiField, S("mu_bis_decl")),
    name="ScalarQCD-bislot-covariant-sum",
    gauge_groups=(QCD_GROUP_BISLOT_SUM,),
)
compiled_bislot_sum = MODEL_BISLOT_SUM_COV.lagrangian().terms
print("Bislotted scalar declaration:", MODEL_BISLOT_SUM_COV.lagrangian_decl)
print("Generated labels:")
for lab in labels(compiled_bislot_sum):
    print(" -", lab)

LEGS_bislot_current = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)
LEGS_bislot_contact = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
    GluonField.leg(p4, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}, species=b4),
)

bislot_species_3 = {b1: phiBidag0, b2: phiBi0, b3: G0}
bislot_species_4 = {b1: phiBidag0, b2: phiBi0, b3: G0, b4: G0}

V_bislot_current_sum = sum(
    (model_vertex(t, LEGS_bislot_current, species_map=bislot_species_3) for t in compiled_bislot_sum if "current" in t.label),
    Expression.num(0),
).expand()
V_bislot_contact_sum = sum(
    (model_vertex(t, LEGS_bislot_contact, species_map=bislot_species_4) for t in compiled_bislot_sum if "contact" in t.label),
    Expression.num(0),
).expand()

show("Bislot scalar summed current", V_bislot_current_sum)
show("Bislot scalar summed contact", V_bislot_contact_sum)


Bislotted scalar declaration: CovD(PhiBi.bar, mu_bis_decl) * CovD(PhiBi, mu_bis_decl)
Generated labels:
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (+)_slot1
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (-)_slot1
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (+)_slot2
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (-)_slot2
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 1,1]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 1,2]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 2,1]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 2,2]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) derivative

=== Bislot scalar summed current ===
-1𝑖*gS*g(cof(3, c1),cof(3, c2))*t(coad(8, a3),cof(3, c3),cof(3, c4))*pcomp(p1,mu)+1𝑖*gS*g(cof(3, c1),cof(3, c2))*t(coad(8, a3),cof(3, c3),cof(3, c4))*pcomp(p2,mu)-1𝑖*gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c

### Unbroken electroweak slice

A minimal unbroken `SU(2)_L x U(1)_Y` matter layer already works through
the same `lagrangian_decl` front end.

Keep the picture small:

- one fermion doublet `L`
- one Higgs-like complex scalar doublet `H`
- the unbroken gauge bosons `W` and `B`

From those declarations we can ask directly for:

- the fermion current `Lbar L W`
- the Higgs hypercharge current `Hdag H B`
- the mixed electroweak contact term `Hdag H W B`

There is still no symmetry breaking here. This is only the unbroken
electroweak matter layer.


In [14]:

mu = S("mu")
g1 = S("g1")
g2 = S("g2")
yL = S("yL")
yH = S("yH")

WField = Field(
    "W",
    spin=1,
    self_conjugate=True,
    symbol=S("W0"),
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)

BField = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B0"),
    indices=(LORENTZ_INDEX,),
)

LDoublet = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": yL},
)

HDoublet = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H0"),
    conjugate_symbol=S("Hdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": yH},
)

WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=WField,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson=BField,
    charge="Y",
)

fermion_model = Model(
    I * LDoublet.bar * Gamma(mu) * CovD(LDoublet, mu),
    name="EW-fermion-demo",
    gauge_groups=(SU2L, U1Y),
)

higgs_model = Model(
    CovD(HDoublet.bar, mu) * CovD(HDoublet, mu),
    name="EW-higgs-demo",
    gauge_groups=(SU2L, U1Y),
)

print("Unbroken electroweak fermion current Γ(Lbar, L, W):")
print(fermion_model.lagrangian().feynman_rule(LDoublet.bar, LDoublet, WField))
print()

print("Unbroken electroweak Higgs current Γ(Hdag, H, B):")
print(higgs_model.lagrangian().feynman_rule(HDoublet.bar, HDoublet, BField))
print()

print("Unbroken electroweak mixed contact Γ(Hdag, H, W, B):")
print(higgs_model.lagrangian().feynman_rule(HDoublet.bar, HDoublet, WField, BField))

Unbroken electroweak fermion current Γ(Lbar, L, W):
1𝑖*g2*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))

Unbroken electroweak Higgs current Γ(Hdag, H, B):
-1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu)+1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu)

Unbroken electroweak mixed contact Γ(Hdag, H, W, B):
2𝑖*g1*g2*yH*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w1),cof(2, w2))


## 6. Pure-gauge sector and Yang-Mills self-interactions

Pure-gauge interactions are now declared through the same front-end:

- `-(1/4) * FieldStrength(G, mu, nu) * FieldStrength(G, mu, nu)`

The compiler then generates:

- the gauge bilinear
- the Yang-Mills cubic vertex
- the Yang-Mills quartic vertex

for the covered non-abelian case.


In [15]:
mu_ym_decl, nu_ym_decl = S("mu_ym_decl", "nu_ym_decl")
MODEL_QCD_GAUGE_COV = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP, mu_ym_decl, nu_ym_decl, S("a_ym_decl")) * FieldStrength(QCD_GROUP, mu_ym_decl, nu_ym_decl, S("a_ym_decl")),
    name="QCDGauge-covariant",
    gauge_groups=(QCD_GROUP,),
)

print("Declared Yang-Mills Lagrangian:", MODEL_QCD_GAUGE_COV.lagrangian_decl)
compiled_ym = MODEL_QCD_GAUGE_COV.lagrangian().terms
print("Number of generated pure-gauge interactions:", len(compiled_ym))
print("Generated labels:")
for label in labels(compiled_ym):
    print(" -", label)

LEGS_two_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
)
LEGS_three_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: rho, COLOR_ADJ_KIND: a5}),
)
LEGS_four_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: rho, COLOR_ADJ_KIND: a5}),
    GluonField.leg(p4, species=b4, labels={LORENTZ_KIND: sigma, COLOR_ADJ_KIND: a6}),
)

compiled_ym_lagrangian = CompiledLagrangian(terms=compiled_ym)
# Match the exact pure-gluon signatures rather than every term with the
# same arity. Generic FieldStrength^2 lowering splits the G^3 rule across
# several raw local monomials, so we intentionally sum the exact-signature
# pieces instead of assuming one hard-coded index or one term per arity.
ym_2g = compiled_ym_lagrangian.matching_terms(GluonField, GluonField, sector="pure_gauge")
ym_3g = compiled_ym_lagrangian.matching_terms(GluonField, GluonField, GluonField, sector="pure_gauge")
ym_4g = compiled_ym_lagrangian.matching_terms(GluonField, GluonField, GluonField, GluonField, sector="pure_gauge")
V_ym_bilinear = sum(
    (model_vertex(t, LEGS_two_gluon, species_map={b1: G0, b2: G0}) for t in ym_2g),
    Expression.num(0),
).expand()
V_ym_cubic = sum(
    (model_vertex(t, LEGS_three_gluon, species_map={b1: G0, b2: G0, b3: G0}) for t in ym_3g),
    Expression.num(0),
).expand()
V_ym_4 = sum(
    (model_vertex(t, LEGS_four_gluon, species_map={b1: G0, b2: G0, b3: G0, b4: G0}) for t in ym_4g),
    Expression.num(0),
).expand()

show("Yang-Mills bilinear from FieldStrength", V_ym_bilinear)
show("Yang-Mills cubic from FieldStrength", V_ym_cubic)
show("Yang-Mills quartic from FieldStrength", V_ym_4)


Declared Yang-Mills Lagrangian: -1/4 * FieldStrength(SU3C, mu_ym_decl, nu_ym_decl, a_ym_decl) * FieldStrength(SU3C, mu_ym_decl, nu_ym_decl, a_ym_decl)
Number of generated pure-gauge interactions: 9
Generated labels:
 - 
 - 
 - 
 - 
 - 
 - 
 - 
 - 
 - 

=== Yang-Mills bilinear from FieldStrength ===
1𝑖*g(mink(4, mu),mink(4, nu))*g(coad(8, a3),coad(8, a4))*pcomp(p1,mu1_int)*pcomp(p2,mu1_int)-1𝑖/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu),mink(4, mu1_int))*g(mink(4, nu),mink(4, mu2_int))*pcomp(p1,mu2_int)*pcomp(p2,mu1_int)-1𝑖/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu),mink(4, mu2_int))*g(mink(4, nu),mink(4, mu1_int))*pcomp(p1,mu1_int)*pcomp(p2,mu2_int)

=== Yang-Mills cubic from FieldStrength ===
1/2*gS*g(mink(4, mu),mink(4, nu))*g(mink(4, rho),mink(4, mu1_int))*f(coad(8, a3),coad(8, a4),coad(8, a5))*pcomp(p1,mu1_int)-1/2*gS*g(mink(4, mu),mink(4, nu))*g(mink(4, rho),mink(4, mu1_int))*f(coad(8, a3),coad(8, a5),coad(8, a4))*pcomp(p1,mu1_int)+1/2*gS*g(mink(4, mu),mink(4, nu))*g(mink(4, rho),m

## 6.5 Ordinary Gauge Fixing And Ghosts

Gauge fixing and ghosts now use typed declarative wrappers too:

- `GaugeFixing(G, xi=...)`
- `GhostLagrangian(G)`

These wrappers stay visible in the source declaration and lower onto the same
backend as the older helper classes.


In [16]:
from lagrangian.operators import gauge_fixing_bilinear, gauge_kinetic_bilinear, ghost_gauge, ghost_kinetic

xiQED, xiQCD = S("xiQED", "xiQCD")

MODEL_QED_GAUGE_FIXING = Model(
    GaugeFixing(QED_GROUP, xi=xiQED),
    name="QEDGaugeFixing-covariant",
    gauge_groups=(QED_GROUP,),
)
compiled_qed_gf = MODEL_QED_GAUGE_FIXING.lagrangian().terms
print("QED gauge-fixing declaration:", MODEL_QED_GAUGE_FIXING.lagrangian_decl)
print("QED gauge-fixing terms:", labels(compiled_qed_gf))

LEGS_qed_gf = (
    GaugeField.leg(p1, species=b1, labels={LORENTZ_KIND: mu3}),
    GaugeField.leg(p2, species=b2, labels={LORENTZ_KIND: mu4}),
)
V_qed_gf = model_vertex(compiled_qed_gf[0], LEGS_qed_gf, species_map={b1: A0, b2: A0})
show("Gauge fixing (QED)", V_qed_gf)
show("Compact form", (I / xiQED) * gauge_fixing_bilinear(mu3, mu4, p1, p2))

MODEL_QCD_GAUGE_FIXING = Model(
    GaugeFixing(QCD_GROUP, xi=xiQCD),
    name="QCDGaugeFixing-covariant",
    gauge_groups=(QCD_GROUP,),
)
compiled_qcd_gf = MODEL_QCD_GAUGE_FIXING.lagrangian().terms
print("QCD gauge-fixing declaration:", MODEL_QCD_GAUGE_FIXING.lagrangian_decl)
print("QCD gauge-fixing terms:", labels(compiled_qcd_gf))

LEGS_qcd_gf = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}),
)
V_qcd_gf = model_vertex(compiled_qcd_gf[0], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
show("Gauge fixing (QCD)", V_qcd_gf)
show("Compact form", (I / xiQCD) * gauge_fixing_bilinear(mu3, mu4, p1, p2) * COLOR_ADJ_INDEX.representation.g(a3, a4).to_expression())

MODEL_QED_GAUGE_FIXED = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(QED_GROUP, mu, nu) * FieldStrength(QED_GROUP, mu, nu) + GaugeFixing(QED_GROUP, xi=xiQED),
    name="QEDGaugeFixed-covariant",
    gauge_groups=(QED_GROUP,),
)
compiled_qed_gauge_fixed = MODEL_QED_GAUGE_FIXED.lagrangian().terms
V_qed_gauge_fixed = simplify_gamma_chain(
    sum((model_vertex(t, LEGS_qed_gf, species_map={b1: A0, b2: A0}) for t in compiled_qed_gauge_fixed), Expression.num(0))
).expand()
photon_rho = compiled_qed_gauge_fixed[0].derivatives[0].lorentz_index
show("Ordinary gauge-fixed photon bilinear", V_qed_gauge_fixed)
show("Compact form", I * (gauge_kinetic_bilinear(mu3, mu4, p1, p2, photon_rho) + gauge_fixing_bilinear(mu3, mu4, p1, p2) / xiQED))

MODEL_QCD_GHOST = Model(
    GhostLagrangian(QCD_GROUP),
    name="QCDGhost-covariant",
    gauge_groups=(QCD_GROUP,),
)
compiled_qcd_ghost = MODEL_QCD_GHOST.lagrangian().terms
print("QCD ghost declaration:", MODEL_QCD_GHOST.lagrangian_decl)
print("QCD ghost terms:", labels(compiled_qcd_ghost))

LEGS_ghost_bilinear = (
    GhostGluonField.leg(p1, conjugated=True, species=b1, labels={COLOR_ADJ_KIND: a3}),
    GhostGluonField.leg(p2, species=b2, labels={COLOR_ADJ_KIND: a4}),
)
LEGS_ghost_gluon = (
    GhostGluonField.leg(p1, conjugated=True, species=b1, labels={COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a4}),
    GhostGluonField.leg(p3, species=b3, labels={COLOR_ADJ_KIND: a5}),
)
V_ghost_bilinear = model_vertex(compiled_qcd_ghost[0], LEGS_ghost_bilinear, species_map={b1: ghGbar0, b2: ghG0})
V_ghost_gluon = model_vertex(compiled_qcd_ghost[1], LEGS_ghost_gluon, species_map={b1: ghGbar0, b2: G0, b3: ghG0})
show("Ghost bilinear", V_ghost_bilinear)
show("Compact form", -I * ghost_kinetic(a3, a4, p1, p2, S("rho_ghost")))
show("Ghost-gluon interaction", V_ghost_gluon)
show("Compact form", -gS * ghost_gauge(a3, a4, a5, mu3, p1))

MODEL_QCD_GAUGE_FIXED = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP, mu, nu, S("aC")) * FieldStrength(QCD_GROUP, mu, nu, S("aC")) + GaugeFixing(QCD_GROUP, xi=xiQCD) + GhostLagrangian(QCD_GROUP),
    name="QCDGaugeFixed-covariant",
    gauge_groups=(QCD_GROUP,),
)
compiled_qcd_gauge_fixed = MODEL_QCD_GAUGE_FIXED.lagrangian().terms
qcd_gf_gluon_bilinears = [
    t
    for t in compiled_qcd_gauge_fixed
    if len(t.fields) == 2 and all(occ.field is GluonField for occ in t.fields)
]
V_qcd_gauge_fixed = simplify_gamma_chain(
    sum((model_vertex(t, LEGS_qcd_gf, species_map={b1: G0, b2: G0}) for t in qcd_gf_gluon_bilinears), Expression.num(0))
).expand()
qcd_gf_derivative_term = next((t for t in qcd_gf_gluon_bilinears if t.derivatives), None)
if qcd_gf_derivative_term is None:
    raise ValueError("Expected at least one derivative gluon bilinear in compiled_qcd_gauge_fixed.")
gluon_rho = qcd_gf_derivative_term.derivatives[0].lorentz_index
show("Ordinary gauge-fixed gluon bilinear", V_qcd_gauge_fixed)
show("Compact form", I * (gauge_kinetic_bilinear(mu3, mu4, p1, p2, gluon_rho) + gauge_fixing_bilinear(mu3, mu4, p1, p2) / xiQCD) * COLOR_ADJ_INDEX.representation.g(a3, a4).to_expression())


QED gauge-fixing declaration: GaugeFixing(U1QED, xi=xiQED)
QED gauge-fixing terms: ['-(1/2 xiQED) (U1QED gauge fixing)']

=== Gauge fixing (QED) ===
1𝑖*(1/2*g(mink(4, mu3),mink(4, mu1_int))*g(mink(4, mu4),mink(4, mu2_int))*pcomp(p1,mu1_int)*pcomp(p2,mu2_int)/xiQED+1/2*g(mink(4, mu3),mink(4, mu2_int))*g(mink(4, mu4),mink(4, mu1_int))*pcomp(p1,mu2_int)*pcomp(p2,mu1_int)/xiQED)

=== Compact form ===
1𝑖*pcomp(p1,mu4)*pcomp(p2,mu3)/xiQED
QCD gauge-fixing declaration: GaugeFixing(SU3C, xi=xiQCD)
QCD gauge-fixing terms: ['-(1/2 xiQCD) (SU3C gauge fixing)']

=== Gauge fixing (QCD) ===
1𝑖*(1/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu3),mink(4, mu1_int))*g(mink(4, mu4),mink(4, mu2_int))*pcomp(p1,mu1_int)*pcomp(p2,mu2_int)/xiQCD+1/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu3),mink(4, mu2_int))*g(mink(4, mu4),mink(4, mu1_int))*pcomp(p1,mu2_int)*pcomp(p2,mu1_int)/xiQCD)

=== Compact form ===
1𝑖*g(coad(8, a3),coad(8, a4))*pcomp(p1,mu4)*pcomp(p2,mu3)/xiQCD

=== Ordinary gauge-fixed photon bilinear ===
1

## 7. Recommended `lagrangian_decl` API

Use `lagrangian_decl` as a sum of source terms.

1. define `Field`, `GaugeGroup`, and optional `GaugeRepresentation` metadata
2. write each source term directly in the front-end syntax
3. call `Model(...).lagrangian()` to compile the declaration
4. extract the vertex you want with `feynman_rule(...)`

Supported front-end building blocks:

- local monomials such as `lam * Phi * Phi * Phi * Phi`
- local derivative monomials through `PartialD(...)`, for example `PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi`
- local fermion derivative operators such as `Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi`
- fermions: `I * Psi.bar * Gamma(mu) * CovD(Psi, mu)`
- complex scalars: `CovD(Phi.bar, mu) * CovD(Phi, mu)`
- dressed covariant terms such as `I * Psi.bar * Gamma(mu) * CovD(Psi, mu) * Phi * Phi`
- pure gauge: `-(1/4) * FieldStrength(G, mu, nu) * FieldStrength(G, mu, nu)`
- gauge fixing: `GaugeFixing(G, xi=...)`
- ghosts: `GhostLagrangian(G)`

Internally, the declaration is compiled one source term at a time. Purely local
terms lower directly to backend compiled interactions, while only terms containing
canonical `CovD`, `FieldStrength`, gauge-fixing, or ghost structure are expanded.

Some exotic local tensor structures are not yet covered by the recommended declarative
front-end. Treat any backend-only fallback as an internal escape hatch, not as the modern public API.


In [17]:
# Scalar: a generic local operator can be declared directly as a field product.
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
lam_phi4 = S("lam_phi4")
MODEL_SCALAR4 = Model(lam_phi4*Phi*Phi*Phi*Phi)
L_scalar = MODEL_SCALAR4.lagrangian()

print("Source declaration:", MODEL_SCALAR4.lagrangian_decl)
show("Scalar vertex phi phi phi phi", L_scalar.feynman_rule(Phi, Phi, Phi, Phi))


Source declaration: lam_phi4 * Phi * Phi * Phi * Phi

=== Scalar vertex phi phi phi phi ===
24𝑖*lam_phi4


In [18]:
# Several source terms can be summed in one declaration; each term is compiled separately.
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
m2 = S("m2")
lam_phi4 = S("lam_phi4")
MODEL_SUM = Model(
    -m2 * Phi * Phi + lam_phi4 * Phi * Phi * Phi * Phi,
)
L_sum = MODEL_SUM.lagrangian()

print("Source declaration:", MODEL_SUM.lagrangian_decl)
show("Mass vertex phi phi", L_sum.feynman_rule(Phi, Phi))
show("Quartic vertex phi phi phi phi", L_sum.feynman_rule(Phi, Phi, Phi, Phi))


Source declaration: -m2 * Phi * Phi + lam_phi4 * Phi * Phi * Phi * Phi

=== Mass vertex phi phi ===
-2𝑖*m2

=== Quartic vertex phi phi phi phi ===
24𝑖*lam_phi4


In [19]:
# Fermion: the canonical covariant kinetic term generates the gauge current.
mu = S("mu")
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": S("Qpsi")},
)
A = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
U1 = GaugeGroup(name="U1", abelian=True, coupling=S("e"), gauge_boson=A, charge="Q")
MODEL_FERMION = Model(
    I * Psi.bar * Gamma(mu) * CovD(Psi, mu),
    gauge_groups=(U1,),
)
L_fermion = MODEL_FERMION.lagrangian()

print("Source declaration:", MODEL_FERMION.lagrangian_decl)
show("Fermion vertex psibar psi A", L_fermion.feynman_rule(Psi.bar, Psi, A))


Source declaration: 1𝑖 * Psi.bar * Gamma(mu) * CovD(Psi, mu)

=== Fermion vertex psibar psi A ===
1𝑖*Qpsi*e*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


In [20]:
# Complex scalar: the canonical covariant kinetic term generates both current and contact pieces.
mu = S("mu")
PhiC = Field(
    "PhiC",
    spin=0,
    self_conjugate=False,
    symbol=S("phiC"),
    conjugate_symbol=S("phiCbar"),
    quantum_numbers={"Q": S("Qphi")},
)
A = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
U1 = GaugeGroup(name="U1", abelian=True, coupling=S("e"), gauge_boson=A, charge="Q")
MODEL_SCALAR_GAUGE = Model(
    CovD(PhiC.bar, mu) * CovD(PhiC, mu),
    gauge_groups=(U1,),
)
L_scalar_gauge = MODEL_SCALAR_GAUGE.lagrangian()

print("Source declaration:", MODEL_SCALAR_GAUGE.lagrangian_decl)
show("Complex-scalar current phi^dag phi A", L_scalar_gauge.feynman_rule(PhiC.bar, PhiC, A))
show("Complex-scalar contact phi^dag phi A A", L_scalar_gauge.feynman_rule(PhiC.bar, PhiC, A, A))


Source declaration: CovD(PhiC.bar, mu) * CovD(PhiC, mu)

=== Complex-scalar current phi^dag phi A ===
-1𝑖*e*Qphi*pcomp(q1,mu)+1𝑖*e*Qphi*pcomp(q2,mu)

=== Complex-scalar contact phi^dag phi A A ===
2𝑖*e^2*Qphi^2*g(mink(4, mu3),mink(4, mu4))


In [21]:
# Derivatives: ordinary local derivative operators can be declared with PartialD(...).
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
mu = S("mu")
MODEL_DERIV = Model(
    S("gD2") * PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi,
)
L_deriv = MODEL_DERIV.lagrangian()

print("Source declaration:", MODEL_DERIV.lagrangian_decl)
show("Derivative vertex (d phi)^2 phi^2", L_deriv.feynman_rule(Phi, Phi, Phi, Phi))


Source declaration: gD2 * PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi

=== Derivative vertex (d phi)^2 phi^2 ===
-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q3,mu1_int)*pcomp(q4,mu1_int)


### Local derivatives through `PartialD(...)`

`PartialD(...)` is the declarative front end for ordinary local derivatives.
It is not a gauge-covariant derivative. It just marks which field the ordinary
partial derivative acts on inside a local monomial.

Typical patterns are:

- `PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi`
- `Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi`
- nested ordinary derivatives such as `PartialD(PartialD(Phi, mu), nu)` when needed


In [22]:
# Local fermion derivative: PartialD(...) also works in ordinary fermion operators.
mu, nu = S("mu"), S("nu")
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
)
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
Chi = Field("Chi", spin=0, self_conjugate=True, symbol=S("chi"))
MODEL_LOCAL_FERMION_DERIV = Model(
    S("g_local") * Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi,
)
L_local_fermion_deriv = MODEL_LOCAL_FERMION_DERIV.lagrangian()

print("Source declaration:", MODEL_LOCAL_FERMION_DERIV.lagrangian_decl)
show(
    "Local fermion derivative psibar d psi phi chi",
    L_local_fermion_deriv.feynman_rule(Psi.bar, Psi, Phi, Chi),
)


Source declaration: g_local * Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi

=== Local fermion derivative psibar d psi phi chi ===
g_local*gamma(bis(4, i1),bis(4, i2),mink(4, mu))*pcomp(q2,mu1_int)


In [23]:
# Dressed covariant operator: the same source term yields a derivative vertex and a gauge-current vertex.
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": S("Qpsi")},
)
A = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
U1 = GaugeGroup(name="U1", abelian=True, coupling=S("e"), gauge_boson=A, charge="Q")
mu = S("mu")
MODEL_DRESSED = Model(
    I * Psi.bar * Gamma(mu) * CovD(Psi, mu) * Phi * Phi,
    gauge_groups=(U1,),
)
L_dressed = MODEL_DRESSED.lagrangian()

print("Source declaration:", MODEL_DRESSED.lagrangian_decl)
show("Dressed operator: psibar d psi phi phi", L_dressed.feynman_rule(Psi.bar, Psi, Phi, Phi))
show("Dressed operator: psibar psi A phi phi", L_dressed.feynman_rule(Psi.bar, Psi, A, Phi, Phi))


Source declaration: 1𝑖 * Psi.bar * Gamma(mu) * CovD(Psi, mu) * Phi * Phi

=== Dressed operator: psibar d psi phi phi ===
2𝑖*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

=== Dressed operator: psibar psi A phi phi ===
2𝑖*Qpsi*e*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


In [24]:
# Gauge fixing: use the typed wrapper for the ordinary linear covariant gauge term.
B = Field("B", spin=1, self_conjugate=True, symbol=S("B"), indices=(LORENTZ_INDEX,))
U1GF = GaugeGroup(name="U1GF", abelian=True, coupling=S("eGF"), gauge_boson=B, charge="Q")
MODEL_GF = Model(
    GaugeFixing(U1GF, xi=S("xi")),
    gauge_groups=(U1GF,),
)
L_gf = MODEL_GF.lagrangian()

print("Source declaration:", MODEL_GF.lagrangian_decl)
show("Gauge-fixing bilinear A A", L_gf.feynman_rule(B, B))


Source declaration: GaugeFixing(U1GF, xi=xi)

=== Gauge-fixing bilinear A A ===
1𝑖*pcomp(q1,mu1)*pcomp(q2,mu2)/xi


In [25]:
# Ghosts: use the typed wrapper for the ordinary Faddeev-Popov sector.
MODEL_GHOST = Model(
    GhostLagrangian(QCD_GROUP),
    gauge_groups=(QCD_GROUP,),
)
L_ghost = MODEL_GHOST.lagrangian()

print("Source declaration:", MODEL_GHOST.lagrangian_decl)
show("Ghost bilinear cbar c", L_ghost.feynman_rule(GhostGluonField.bar, GhostGluonField))
show("Ghost-gluon vertex cbar G c", L_ghost.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField))


Source declaration: GhostLagrangian(SU3C)

=== Ghost bilinear cbar c ===
-1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

=== Ghost-gluon vertex cbar G c ===
gS*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)


The wrapper above is the recommended front-end syntax. The next cell is an implementation-level equivalence check: it writes the same ordinary gauge-fixing and Faddeev-Popov ghost source terms directly with raw Spenso tensors. The explicit color and Lorentz metrics bind chosen field-slot labels to derivative indices, and the result should lower to the same compiled interactions as `GaugeFixing(QCD_GROUP, xi=xiQCD) + GhostLagrangian(QCD_GROUP)`.

In [26]:
# Advanced check: write the same gauge-fixing and ghost sector with raw tensor coefficients.
mu_left = S("mu_left")
mu_right = S("mu_right")
mu_gluon = S("mu_gluon")
mu_ghost = S("mu")
rho_left = S("rho_left")
rho_right = S("rho_right")
rho_ghost = S("rho_ghost")
a_left = S("a_left")
a_right = S("a_right")
a_bar = S("a_bar")
a_gluon = S("a_gluon")
a_ghost = S("a_ghost")

MODEL_GHOST_MANUAL = Model(
    (
        -(Expression.num(1) / (Expression.num(2) * xiQCD))
        * COLOR_ADJ_INDEX.representation.g(a_left, a_right).to_expression()
        * LORENTZ_INDEX.representation.g(mu_left, rho_left).to_expression()
        * LORENTZ_INDEX.representation.g(mu_right, rho_right).to_expression()
        * PartialD(GluonField, rho_left, labels={LORENTZ_KIND: mu_left, COLOR_ADJ_KIND: a_left})
        * PartialD(GluonField, rho_right, labels={LORENTZ_KIND: mu_right, COLOR_ADJ_KIND: a_right})
        + COLOR_ADJ_INDEX.representation.g(a_bar, a_ghost).to_expression()
        * PartialD(GhostGluonField, mu_ghost, conjugated=True, labels={COLOR_ADJ_KIND: a_bar})
        * PartialD(GhostGluonField, mu_ghost, labels={COLOR_ADJ_KIND: a_ghost})
        + (
            -gS
            * structure_constant(a_bar, a_gluon, a_ghost)
            * LORENTZ_INDEX.representation.g(rho_ghost, mu_gluon).to_expression()
            * PartialD(GhostGluonField, rho_ghost, conjugated=True, labels={COLOR_ADJ_KIND: a_bar})
            * GluonField(mu_gluon, a_gluon)
            * GhostGluonField(a_ghost)
        )
    ),
    gauge_groups=(QCD_GROUP,),
)
MODEL_GHOST_WRAPPER = Model(
    GaugeFixing(QCD_GROUP, xi=xiQCD) + GhostLagrangian(QCD_GROUP),
    gauge_groups=(QCD_GROUP,),
)

L_ghost_manual = MODEL_GHOST_MANUAL.lagrangian()
L_ghost_wrapper = MODEL_GHOST_WRAPPER.lagrangian()

print("Manual declaration:", MODEL_GHOST_MANUAL.lagrangian_decl)
show("Manual gauge-fixing bilinear G G", L_ghost_manual.feynman_rule(GluonField, GluonField, simplify=True))
show("Manual ghost-gluon vertex cbar G c", L_ghost_manual.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField, simplify=True))

manual_vs_wrapper = {
    "gauge fixing": L_ghost_manual.feynman_rule(GluonField, GluonField, simplify=True).expand().to_canonical_string()
    == L_ghost_wrapper.feynman_rule(GluonField, GluonField, simplify=True).expand().to_canonical_string(),
    "ghost bilinear": L_ghost_manual.feynman_rule(GhostGluonField.bar, GhostGluonField, simplify=True).expand().to_canonical_string()
    == L_ghost_wrapper.feynman_rule(GhostGluonField.bar, GhostGluonField, simplify=True).expand().to_canonical_string(),
    "ghost-gluon": L_ghost_manual.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField, simplify=True).expand().to_canonical_string()
    == L_ghost_wrapper.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField, simplify=True).expand().to_canonical_string(),
}
manual_vs_wrapper


Manual declaration: -1/2*g(mink(4, mu_left),mink(4, rho_left))*g(mink(4, mu_right),mink(4, rho_right))*g(coad(8, a_left),coad(8, a_right))/xiQCD * PartialD(G, rho_left) * PartialD(G, rho_right) + g(coad(8, a_bar),coad(8, a_ghost)) * PartialD(ghG.bar, mu) * PartialD(ghG, mu) + -gS*g(mink(4, mu_gluon),mink(4, rho_ghost))*f(coad(8, a_bar),coad(8, a_gluon),coad(8, a_ghost)) * PartialD(ghG.bar, rho_ghost) * G * ghG

=== Manual gauge-fixing bilinear G G ===
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

=== Manual ghost-gluon vertex cbar G c ===
-gS*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)


{'gauge fixing': True, 'ghost bilinear': True, 'ghost-gluon': False}

In [27]:
# Yang-Mills: FieldStrength^2 generates the pure-gauge sector.
muYM, nuYM = S("muYM", "nuYM")
MODEL_YM = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP, muYM, nuYM, S("aYM")) * FieldStrength(QCD_GROUP, muYM, nuYM, S("aYM")),
    gauge_groups=(QCD_GROUP,),
)
L_ym = MODEL_YM.lagrangian()

print("Source declaration:", MODEL_YM.lagrangian_decl)
show("Yang-Mills cubic vertex G G G", L_ym.feynman_rule(GluonField, GluonField, GluonField))
show("Yang-Mills quartic vertex G G G G", L_ym.feynman_rule(GluonField, GluonField, GluonField, GluonField))


Source declaration: -1/4 * FieldStrength(SU3C, muYM, nuYM, aYM) * FieldStrength(SU3C, muYM, nuYM, aYM)

=== Yang-Mills cubic vertex G G G ===
-gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q1,mu2)+gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu2)+gS*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu1)-gS*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu1)+gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q1,mu3)-gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu3)

=== Yang-Mills quartic vertex G G G G ===
-1𝑖*gS^2*g(mink(4, mu3),mink(4, mu4))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, aYM))*f(coad(8, a4),coad(8, a2),coad(8, aYM))-1𝑖*gS^2*g(mink(4, mu3),mink(4, mu4))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a2),coad(8, aYM))*f(coad(8, a4),coad(8, a1),coad(8, aYM))-1𝑖*gS^2*g(m

In [28]:
# Full gauge-fixed QCD sector: combine pure gauge, gauge fixing, and ghosts in one declaration.
muQCD, nuQCD = S("muQCD", "nuQCD")
MODEL_QCD_FULL = Model(
    (
        -(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP, muQCD, nuQCD, S("aQCD")) * FieldStrength(QCD_GROUP, muQCD, nuQCD, S("aQCD"))
        + GaugeFixing(QCD_GROUP, xi=xiQCD)
        + GhostLagrangian(QCD_GROUP)
    ),
    gauge_groups=(QCD_GROUP,),
)
L_qcd_full = MODEL_QCD_FULL.lagrangian()

print("Source declaration:", MODEL_QCD_FULL.lagrangian_decl)
show("Gauge-fixed gluon bilinear", L_qcd_full.feynman_rule(GluonField, GluonField))
show("Three-gluon vertex", L_qcd_full.feynman_rule(GluonField, GluonField, GluonField))
show("Ghost-gluon vertex", L_qcd_full.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField))


Source declaration: -1/4 * FieldStrength(SU3C, muQCD, nuQCD, aQCD) * FieldStrength(SU3C, muQCD, nuQCD, aQCD) + GaugeFixing(SU3C, xi=xiQCD) + GhostLagrangian(SU3C)

=== Gauge-fixed gluon bilinear ===
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD+1𝑖*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu2)*pcomp(q2,mu1)

=== Three-gluon vertex ===
-gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q1,mu2)+gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu2)+gS*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu1)-gS*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu1)+gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q1,mu3)-gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu3)

=== Ghost-gluon vertex ===
gS*f(coad(8, a1),

In [29]:
# SM slice: the same source term yields the derivative piece plus the gluon and photon currents.
# With identical Phi * Phi spectators, the vertices carry the expected combinatorial factor 2.
muSM = S("muSM")
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("Phi"))
qSM = Field(
    "qSM",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("qSM0"),
    conjugate_symbol=S("qSMbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": S("QqSM")},
)
MODEL_SM_SLICE = Model(
    I * qSM.bar * Gamma(muSM) * CovD(qSM, muSM) * Phi * Phi,
    gauge_groups=(QCD_GROUP, QED_GROUP),
)
L_sm = MODEL_SM_SLICE.lagrangian()

print("Source declaration:", MODEL_SM_SLICE.lagrangian_decl)
show("SM slice: qbar q Phi Phi", L_sm.feynman_rule(qSM.bar, qSM, Phi, Phi))
show("SM slice: qbar q G Phi Phi", L_sm.feynman_rule(qSM.bar, qSM, GluonField, Phi, Phi))
show("SM slice: qbar q A Phi Phi", L_sm.feynman_rule(qSM.bar, qSM, GaugeField, Phi, Phi))


Source declaration: 1𝑖 * qSM.bar * Gamma(muSM) * CovD(qSM, muSM) * Phi * Phi

=== SM slice: qbar q Phi Phi ===
2𝑖*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

=== SM slice: qbar q G Phi Phi ===
2𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

=== SM slice: qbar q A Phi Phi ===
2𝑖*eQED*QqSM*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


## 8. Current coverage

Recommended workflow:

1. define metadata
2. write the declaration directly inside `Model(...)`
3. call `Model.lagrangian()`
4. ask for the vertex you want with `feynman_rule(...)`

Implemented declarative coverage:

- local monomials and sums of local monomials
- ordinary local derivative monomials through `PartialD(...)`, including nested ordinary derivatives
- canonical fermion and complex-scalar covariant terms through `CovD(...)`
- dressed covariant operators with extra local spectators
- pure-gauge Yang-Mills terms through `FieldStrength(...)`
- linear covariant gauge fixing through `GaugeFixing(...)`
- non-abelian ghosts through `GhostLagrangian(...)`
- mixed `SU(3)_c x U(1)` models in one declaration

Current limits:

- there is no free-form symbolic parser yet; only the supported canonical patterns expand automatically
- `feynman_rule(...)` extracts one chosen vertex, not yet a whole-Lagrangian `FeynmanRules[...]` equivalent
- some exotic local tensor structures are not yet covered by the declarative front-end
- repeated same-kind gauge slots may still need explicit `slot_policy` choices
- the full electroweak Standard Model and background-field layer are still future work

For the internals, start with `src/model/`, `src/compiler/gauge.py`, and `src/symbolic/vertex_engine.py`.


## 9. What to use for full regression coverage

This notebook is a front-end walkthrough, not the full validation matrix.

For exhaustive executable coverage use:

- `examples/examples_lagrangian.py` for declarative demos
- `examples/examples.py` for the broader reference suite
- `tests/` for focused API and compiler regressions
